# 🔥 Aumento de complejidad en PySpark con múltiples lecturas y uniones
Este notebook demuestra cómo la complejidad del plan de ejecución de Spark aumenta cuando:
- Se generan múltiples versiones de un Parquet.
- Se leen y unen en un bucle.
- Se realiza una agregación compleja sobre el dataset combinado.

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Crear sesión Spark
spark = SparkSession.builder.appName('ComplejidadUnionParquet').getOrCreate()

# Ruta base del parquet original
base_path = 'hdfs://192.168.1.144:8020/data/salida_indicadores_pnd_extenso.parquet'

# Leer parquet base
df_base = spark.read.parquet(base_path)
df_base.show(5)

+-------+----+------+--------------------+---------------------+--------------------+--------------------+---------------------+----------------------+----------------------+---------------------+--------+----------+---------+--------------+---------------+---------------+------------------------+---------+-----------+--------------+-------------------------------+-----------------------------------------+--------------------------------+---------------------+-----+-----------+------------------+--------------------+------------+--------+-------+-----------------------+-------------------------+--------------------+--------------------+-------------------------+--------------------------+--------------------------+-------------------------+------------+----------+-------------+------------------+-------------------+-------------------+----------------------------+-------------+---------------+------------------+-----------------------------------+---------------------------------------

In [15]:
df_base.count()

1361

## 1️⃣ Generar múltiples versiones del Parquet para simular datos de diferentes fuentes

In [17]:
# Crear múltiples versiones con ligeras modificaciones y guardarlas en subcarpetas
for i in range(1, 4):
    df_mod = df_base
    output_version_path = f'/var/snp-dwh/sftp/output/version_{i}'
    df_mod.write.mode('overwrite').parquet(output_version_path)

print('✅ Se generaron versiones v1, v2 y v3 del parquet')

✅ Se generaron versiones v1, v2 y v3 del parquet


## 2️⃣ Leer todas las versiones y unirlas en un bucle

In [18]:
paths = [f'/var/snp-dwh/sftp/output/version_{i}' for i in range(1, 4)]

# Leer el primer parquet como base
df_union = spark.read.parquet(paths[0])

# Unir el resto en un bucle
for p in paths[1:]:
    df_next = spark.read.parquet(p)
    df_union = df_union.unionByName(df_next)

print('✅ Unión de todas las versiones completada')
df_union.show(10)

✅ Unión de todas las versiones completada
+-------+----+------+--------------------+---------------------+--------------------+--------------------+---------------------+----------------------+----------------------+---------------------+--------+----------+------------------+--------------+---------------+---------------+------------------------+----------+------------+--------------+-------------------------------+-----------------------------------------+--------------------------------+---------------------+-----+-----------+------------------+--------------------+------------+--------+-------+-----------------------+-------------------------+--------------------+--------------------+-------------------------+--------------------------+--------------------------+-------------------------+------------+----------+------------------+------------------+-------------------+-------------------+----------------------------+-------------+---------------+------------------+-----------------

## 3️⃣ Operación adicional: agregación compleja tras la unión

In [19]:
df_complex = df_union.groupBy('nombre_provincia') \
    .agg(
        F.sum('total_habitantes').alias('habitantes_acumulados'),
        F.avg('total_habitantes').alias('habitantes_promedio'),
        F.countDistinct('version').alias('fuentes')
    ) \
    .orderBy(F.desc('habitantes_acumulados'))

df_complex.show()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `nombre_provincia` cannot be resolved. Did you mean one of the following? [`codigo_`, `EJE_AUX`, `ERROR_ESTANDAR`, `FECHA_AUX`, `MES_ANIO`].;
'Aggregate ['nombre_provincia], ['nombre_provincia, sum('total_habitantes) AS habitantes_acumulados#2363, avg('total_habitantes) AS habitantes_promedio#2365, 'count(distinct 'version) AS fuentes#2366]
+- Union false, false
   :- Relation [codigo_#1534,TIPO#1535,EJE#1536,NOMBRE_DEL_OBJETIVO#1537,NOMBRE_DE_LA_POLITICA#1538,META#1539,INDICADOR#1540,FUENTE_DE_INFORMACION#1541,GRUPO_DE_DESAGREGACION#1542,NIVEL_DE_DESAGREGACION#1543,CODIGO_GEOGRAFICO_DPA#1544,MES_ANIO#1545,FECHA#1546,ESTIMADOR#1547,ERROR_ESTANDAR#1548,LIMITE_INFERIOR#1549,LIMITE_SUPERIOR#1550,COEFICIENTE_DE_VARIACION#1551,NUMERADOR#1552,DENOMINADOR#1553,NOMBRE_DEL_EJE#1554,PERIODICIDAD_FICHA_METODOLOGICA#1555,FECHA_DE_TRANSFERENCIA_FICHA_METODOLOGICA#1556,DESAGREGACION_FICHA_METODOLOGICA#155

## 4️⃣ Visualización de la complejidad del plan de ejecución

In [ ]:
# Mostrar el plan de ejecución para observar cómo Spark maneja múltiples lecturas y uniones
df_complex.explain(True)

== Parsed Logical Plan ==
'Sort ['habitantes_acumulados DESC NULLS LAST], true
+- Aggregate [nombre_provincia#67], [nombre_provincia#67, sum(total_habitantes#68) AS habitantes_acumulados#131, avg(total_habitantes#68) AS habitantes_promedio#133, count(distinct version#70) AS fuentes#134L]
   +- Union false, false
      :- Relation [nombre_provincia#67,total_habitantes#68,promedio_habitantes#69,version#70] parquet
      :- Project [nombre_provincia#75, total_habitantes#76, promedio_habitantes#77, version#78]
      :  +- Relation [nombre_provincia#75,total_habitantes#76,promedio_habitantes#77,version#78] parquet
      +- Project [nombre_provincia#88, total_habitantes#89, promedio_habitantes#90, version#91]
         +- Relation [nombre_provincia#88,total_habitantes#89,promedio_habitantes#90,version#91] parquet

== Analyzed Logical Plan ==
nombre_provincia: string, habitantes_acumulados: double, habitantes_promedio: double, fuentes: bigint
Sort [habitantes_acumulados#131 DESC NULLS LAST], t